# Interactive Signal Feature Learning

This notebook allows you to visually explore various signal processing features by adjusting parameters interactively.

## Features to Explore:
- **Kurtosis**: Measure of tailedness in the distribution
- **Skewness**: Measure of asymmetry in the distribution
- **Peak Detection**: Finding peaks with adjustable thresholds
- **Filtering**: Low-pass, high-pass, band-pass filters
- **Window Functions**: Different windowing techniques
- **FFT Analysis**: Frequency domain analysis

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.signal import find_peaks, butter, filtfilt, savgol_filter
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Sample Data

Load your PPG signal data from the windowed CSV files.

In [2]:
# Load a sample windowed CSV file
# Adjust the path to point to your actual data file
data_path = r"C:\Users\DELL\Documents\GitHub\fyp\02_Python_Data_Logger\windowed_data\window_0.csv"

try:
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
except FileNotFoundError:
    print(f"File not found. Please update the data_path variable.")
    # Create synthetic data for demonstration
    t = np.linspace(0, 10, 1000)
    ir_signal = 50000 + 5000 * np.sin(2 * np.pi * 1.2 * t) + np.random.normal(0, 500, len(t))
    red_signal = 45000 + 4000 * np.sin(2 * np.pi * 1.2 * t + 0.5) + np.random.normal(0, 400, len(t))
    df = pd.DataFrame({
        'Timestamp': t,
        'IR_Value': ir_signal,
        'Red_Value': red_signal
    })
    print("Using synthetic data for demonstration.")

File not found. Please update the data_path variable.
Using synthetic data for demonstration.


## 2. Interactive Kurtosis Visualization

Explore how kurtosis changes with different signal characteristics.

In [3]:
def plot_kurtosis_analysis(window_size=100, signal_type='IR_Value'):
    """
    Visualize kurtosis for different window sizes
    """
    signal_data = df[signal_type].values
    
    # Calculate rolling kurtosis
    kurtosis_values = []
    positions = []
    
    for i in range(0, len(signal_data) - window_size, window_size // 4):
        window = signal_data[i:i + window_size]
        kurt = stats.kurtosis(window, fisher=True)
        kurtosis_values.append(kurt)
        positions.append(i + window_size // 2)
    
    # Create subplots
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot 1: Original Signal
    axes[0].plot(signal_data, linewidth=0.8, color='blue', alpha=0.7)
    axes[0].set_title(f'{signal_type} - Original Signal', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Sample Index')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Kurtosis over time
    axes[1].plot(positions, kurtosis_values, linewidth=2, color='red', marker='o', markersize=4)
    axes[1].axhline(y=0, color='green', linestyle='--', label='Normal Distribution (Kurt=0)')
    axes[1].set_title(f'Rolling Kurtosis (Window Size: {window_size})', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Kurtosis Value')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Distribution of the entire signal
    axes[2].hist(signal_data, bins=50, color='purple', alpha=0.7, edgecolor='black')
    overall_kurt = stats.kurtosis(signal_data, fisher=True)
    axes[2].set_title(f'Signal Distribution (Overall Kurtosis: {overall_kurt:.3f})', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Amplitude')
    axes[2].set_ylabel('Frequency')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Kurtosis Statistics:")
    print(f"   Overall Kurtosis: {overall_kurt:.3f}")
    print(f"   Mean Rolling Kurtosis: {np.mean(kurtosis_values):.3f}")
    print(f"   Std Rolling Kurtosis: {np.std(kurtosis_values):.3f}")
    print(f"\n💡 Interpretation:")
    if overall_kurt > 0:
        print(f"   Positive kurtosis ({overall_kurt:.3f}) → Heavy tails, more outliers than normal distribution")
    elif overall_kurt < 0:
        print(f"   Negative kurtosis ({overall_kurt:.3f}) → Light tails, fewer outliers than normal distribution")
    else:
        print(f"   Kurtosis ≈ 0 → Similar to normal distribution")

# Create interactive widget
interact(plot_kurtosis_analysis,
         window_size=widgets.IntSlider(min=50, max=500, step=50, value=100, description='Window Size:'),
         signal_type=widgets.Dropdown(options=['IR_Value', 'Red_Value'], value='IR_Value', description='Signal:'));

interactive(children=(IntSlider(value=100, description='Window Size:', max=500, min=50, step=50), Dropdown(des…

## 3. Interactive Peak Detection

Adjust peak detection parameters to find optimal settings.

In [4]:
def plot_peak_detection(height_percentile=70, distance=20, prominence=500, signal_type='IR_Value'):
    """
    Interactive peak detection with adjustable parameters
    """
    signal_data = df[signal_type].values
    
    # Calculate height threshold from percentile
    height_threshold = np.percentile(signal_data, height_percentile)
    
    # Find peaks
    peaks, properties = find_peaks(signal_data, 
                                   height=height_threshold,
                                   distance=distance,
                                   prominence=prominence)
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Plot 1: Signal with peaks
    axes[0].plot(signal_data, linewidth=0.8, color='blue', alpha=0.7, label='Signal')
    axes[0].plot(peaks, signal_data[peaks], 'ro', markersize=8, label=f'Peaks ({len(peaks)} found)')
    axes[0].axhline(y=height_threshold, color='green', linestyle='--', alpha=0.5, label=f'Height Threshold ({height_percentile}%)')
    axes[0].set_title(f'{signal_type} - Peak Detection', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Sample Index')
    axes[0].set_ylabel('Amplitude')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Peak intervals (Heart Rate Variability)
    if len(peaks) > 1:
        peak_intervals = np.diff(peaks)
        axes[1].plot(peak_intervals, linewidth=2, color='red', marker='o', markersize=4)
        axes[1].set_title(f'Peak-to-Peak Intervals (Mean: {np.mean(peak_intervals):.1f} samples)', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Peak Number')
        axes[1].set_ylabel('Interval (samples)')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'Not enough peaks detected', 
                    ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Peak Detection Results:")
    print(f"   Number of peaks: {len(peaks)}")
    if len(peaks) > 1:
        print(f"   Mean peak interval: {np.mean(peak_intervals):.2f} samples")
        print(f"   Std peak interval: {np.std(peak_intervals):.2f} samples")

# Create interactive widget
interact(plot_peak_detection,
         height_percentile=widgets.IntSlider(min=50, max=95, step=5, value=70, description='Height %:'),
         distance=widgets.IntSlider(min=5, max=100, step=5, value=20, description='Min Distance:'),
         prominence=widgets.IntSlider(min=100, max=2000, step=100, value=500, description='Prominence:'),
         signal_type=widgets.Dropdown(options=['IR_Value', 'Red_Value'], value='IR_Value', description='Signal:'));

interactive(children=(IntSlider(value=70, description='Height %:', max=95, min=50, step=5), IntSlider(value=20…

## 4. Interactive Filtering

Explore different filter types and parameters.

In [5]:
def plot_filtering(filter_type='lowpass', cutoff_freq=5, filter_order=4, signal_type='IR_Value', sampling_rate=100):
    """
    Interactive filtering with adjustable parameters
    """
    signal_data = df[signal_type].values
    
    # Design filter
    nyquist = sampling_rate / 2
    normalized_cutoff = cutoff_freq / nyquist
    
    if normalized_cutoff >= 1:
        print("⚠️ Cutoff frequency too high! Reducing to Nyquist frequency.")
        normalized_cutoff = 0.99
    
    b, a = butter(filter_order, normalized_cutoff, btype=filter_type, analog=False)
    filtered_signal = filtfilt(b, a, signal_data)
    
    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot 1: Original vs Filtered
    axes[0].plot(signal_data, linewidth=0.8, color='blue', alpha=0.5, label='Original')
    axes[0].plot(filtered_signal, linewidth=1.5, color='red', label='Filtered')
    axes[0].set_title(f'{signal_type} - {filter_type.capitalize()} Filter (Cutoff: {cutoff_freq} Hz, Order: {filter_order})', 
                     fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Sample Index')
    axes[0].set_ylabel('Amplitude')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Difference (Removed noise/trend)
    difference = signal_data - filtered_signal
    axes[1].plot(difference, linewidth=0.8, color='green', alpha=0.7)
    axes[1].set_title('Removed Component (Original - Filtered)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Amplitude')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Frequency domain comparison
    fft_original = np.fft.fft(signal_data)
    fft_filtered = np.fft.fft(filtered_signal)
    freqs = np.fft.fftfreq(len(signal_data), 1/sampling_rate)
    
    # Only plot positive frequencies
    positive_freqs = freqs[:len(freqs)//2]
    axes[2].plot(positive_freqs, np.abs(fft_original[:len(freqs)//2]), 
                linewidth=0.8, color='blue', alpha=0.5, label='Original')
    axes[2].plot(positive_freqs, np.abs(fft_filtered[:len(freqs)//2]), 
                linewidth=1.5, color='red', label='Filtered')
    axes[2].axvline(x=cutoff_freq, color='green', linestyle='--', alpha=0.7, label=f'Cutoff: {cutoff_freq} Hz')
    axes[2].set_title('Frequency Domain Comparison', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Frequency (Hz)')
    axes[2].set_ylabel('Magnitude')
    axes[2].set_xlim(0, min(20, sampling_rate/2))  # Show up to 20 Hz or Nyquist
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Filtering Statistics:")
    print(f"   Original signal std: {np.std(signal_data):.2f}")
    print(f"   Filtered signal std: {np.std(filtered_signal):.2f}")
    print(f"   Noise reduction: {(1 - np.std(filtered_signal)/np.std(signal_data))*100:.2f}%")

# Create interactive widget
interact(plot_filtering,
         filter_type=widgets.Dropdown(options=['lowpass', 'highpass', 'bandpass'], value='lowpass', description='Filter Type:'),
         cutoff_freq=widgets.FloatSlider(min=0.5, max=20, step=0.5, value=5, description='Cutoff (Hz):'),
         filter_order=widgets.IntSlider(min=1, max=8, step=1, value=4, description='Order:'),
         signal_type=widgets.Dropdown(options=['IR_Value', 'Red_Value'], value='IR_Value', description='Signal:'),
         sampling_rate=widgets.IntSlider(min=50, max=200, step=10, value=100, description='Sample Rate:'));

interactive(children=(Dropdown(description='Filter Type:', options=('lowpass', 'highpass', 'bandpass'), value=…

## 5. Interactive Skewness Analysis

Visualize signal asymmetry with skewness.

In [6]:
def plot_skewness_analysis(window_size=100, signal_type='IR_Value'):
    """
    Visualize skewness for different window sizes
    """
    signal_data = df[signal_type].values
    
    # Calculate rolling skewness
    skewness_values = []
    positions = []
    
    for i in range(0, len(signal_data) - window_size, window_size // 4):
        window = signal_data[i:i + window_size]
        skew = stats.skew(window)
        skewness_values.append(skew)
        positions.append(i + window_size // 2)
    
    # Create subplots
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot 1: Original Signal
    axes[0].plot(signal_data, linewidth=0.8, color='blue', alpha=0.7)
    axes[0].set_title(f'{signal_type} - Original Signal', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Sample Index')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Skewness over time
    axes[1].plot(positions, skewness_values, linewidth=2, color='orange', marker='o', markersize=4)
    axes[1].axhline(y=0, color='green', linestyle='--', label='Symmetric (Skew=0)')
    axes[1].set_title(f'Rolling Skewness (Window Size: {window_size})', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Skewness Value')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Distribution
    axes[2].hist(signal_data, bins=50, color='teal', alpha=0.7, edgecolor='black')
    overall_skew = stats.skew(signal_data)
    axes[2].set_title(f'Signal Distribution (Overall Skewness: {overall_skew:.3f})', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Amplitude')
    axes[2].set_ylabel('Frequency')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Skewness Statistics:")
    print(f"   Overall Skewness: {overall_skew:.3f}")
    print(f"   Mean Rolling Skewness: {np.mean(skewness_values):.3f}")
    print(f"\n💡 Interpretation:")
    if overall_skew > 0.5:
        print(f"   Positive skewness ({overall_skew:.3f}) → Right-skewed, tail extends to the right")
    elif overall_skew < -0.5:
        print(f"   Negative skewness ({overall_skew:.3f}) → Left-skewed, tail extends to the left")
    else:
        print(f"   Skewness ≈ 0 → Approximately symmetric distribution")

# Create interactive widget
interact(plot_skewness_analysis,
         window_size=widgets.IntSlider(min=50, max=500, step=50, value=100, description='Window Size:'),
         signal_type=widgets.Dropdown(options=['IR_Value', 'Red_Value'], value='IR_Value', description='Signal:'));

interactive(children=(IntSlider(value=100, description='Window Size:', max=500, min=50, step=50), Dropdown(des…

## 6. Save Your Findings

Use this cell to document optimal parameters you discover.

In [7]:
# Document your optimal parameters here
optimal_parameters = {
    'kurtosis_window': 100,
    'peak_detection': {
        'height_percentile': 70,
        'distance': 20,
        'prominence': 500
    },
    'filtering': {
        'type': 'lowpass',
        'cutoff': 5,
        'order': 4
    },
    'notes': 'Add your observations here'
}

print("Optimal Parameters:")
print(optimal_parameters)

Optimal Parameters:
{'kurtosis_window': 100, 'peak_detection': {'height_percentile': 70, 'distance': 20, 'prominence': 500}, 'filtering': {'type': 'lowpass', 'cutoff': 5, 'order': 4}, 'notes': 'Add your observations here'}
